# Scenario Comparison
Compares two simulation runs side-by-side across network flows, chokepoint throughput, top-affected routes, and port-level impact.

In [ ]:
# ── Scenario configuration ────────────────────────────────────────────────────
SCENARIO_A   = 'scenario_baseline'
SCENARIO_B   = 'scenario_suez_50pct_reduction'
LABEL_A      = 'Baseline'
LABEL_B      = 'Suez −50%'

SIM_ROOT     = '../part_4_new_simulation/simulation_output_data'
NETWORK_FILE = '../part_3_network_extraction/network_outputs/network_calibrated.gpickle'

# How many top edges to draw on the network map (avoids overplotting)
TOP_EDGES_MAP    = 800
# How many edge gainers / losers to show in bar chart
TOP_EDGES_BAR    = 20
# Minimum baseline ship count to include an edge in comparison
MIN_SHIPS        = 5

print(f'Comparing  A: {LABEL_A}  vs  B: {LABEL_B}')

In [ ]:
import pickle, warnings
import numpy as np
import pandas as pd
import networkx as nx
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
warnings.filterwarnings('ignore')

DIR_A = f'{SIM_ROOT}/{SCENARIO_A}'
DIR_B = f'{SIM_ROOT}/{SCENARIO_B}'

print('Imports OK')

In [ ]:
# ── Load network graph ────────────────────────────────────────────────────────
with open(NETWORK_FILE, 'rb') as f:
    G = pickle.load(f)

# Node positions: {node_id: (lon, lat)}
pos = {n: (G.nodes[n]['lon'], G.nodes[n]['lat']) for n in G.nodes}

# World basemap
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

# ── Load simulation outputs ───────────────────────────────────────────────────
edge_a = pd.read_parquet(f'{DIR_A}/edge_statistics.parquet')
edge_b = pd.read_parquet(f'{DIR_B}/edge_statistics.parquet')

choke_a = pd.read_parquet(f'{DIR_A}/choke_cargo.parquet')
choke_b = pd.read_parquet(f'{DIR_B}/choke_cargo.parquet')

port_a = pd.read_parquet(f'{DIR_A}/port_cargo.parquet')
port_b = pd.read_parquet(f'{DIR_B}/port_cargo.parquet')

print(f'Edge rows  A={len(edge_a):,}  B={len(edge_b):,}')
print(f'Choke rows A={len(choke_a):,}  B={len(choke_b):,}')
print(f'Port rows  A={len(port_a):,}  B={len(port_b):,}')

In [ ]:
# ── Merge edge statistics ─────────────────────────────────────────────────────
# Normalise direction so (min, max) is always the key
for df in (edge_a, edge_b):
    df['e0'] = df[['node1','node2']].min(axis=1)
    df['e1'] = df[['node1','node2']].max(axis=1)

ea = edge_a[['e0','e1','ship_count','cargo_total_weight','cargo_total_value']].rename(
    columns={'ship_count':'ships_a','cargo_total_weight':'weight_a','cargo_total_value':'value_a'})
eb = edge_b[['e0','e1','ship_count','cargo_total_weight','cargo_total_value']].rename(
    columns={'ship_count':'ships_b','cargo_total_weight':'weight_b','cargo_total_value':'value_b'})

edges = pd.merge(ea, eb, on=['e0','e1'], how='outer').fillna(0)

# Relative change (weight): (B - A) / A; cap at ±5 for colour scale
eps = 1e3  # avoid div/0 on near-zero edges
edges['rel_change'] = (edges['weight_b'] - edges['weight_a']) / (edges['weight_a'] + eps)
edges['abs_change'] = edges['weight_b'] - edges['weight_a']
edges['avg_weight'] = (edges['weight_a'] + edges['weight_b']) / 2

# Attach node positions
edges['lon0'] = edges['e0'].map(lambda n: pos.get(n, (np.nan,np.nan))[0])
edges['lat0'] = edges['e0'].map(lambda n: pos.get(n, (np.nan,np.nan))[1])
edges['lon1'] = edges['e1'].map(lambda n: pos.get(n, (np.nan,np.nan))[0])
edges['lat1'] = edges['e1'].map(lambda n: pos.get(n, (np.nan,np.nan))[1])
edges = edges.dropna(subset=['lon0','lat0','lon1','lat1'])

# Filter: must have meaningful traffic in at least one scenario
active = edges[(edges['ships_a'] >= MIN_SHIPS) | (edges['ships_b'] >= MIN_SHIPS)].copy()
print(f'Active edges (≥{MIN_SHIPS} ships in either scenario): {len(active):,}')

In [ ]:
# ── 1. Network Comparison Map ─────────────────────────────────────────────────
# Edge width  ∝ avg cargo weight across both scenarios
# Edge colour = relative change (dark green = most gained, dark red = most lost)

plot_df = active.nlargest(TOP_EDGES_MAP, 'avg_weight').copy()

# Width scaling
w_max = plot_df['avg_weight'].max()
MIN_W, MAX_W = 0.3, 5.0
plot_df['lw'] = MIN_W + (MAX_W - MIN_W) * np.sqrt(plot_df['avg_weight'] / w_max)

# Colour: symmetric diverging scale capped at ±100%
CMAP   = cm.RdYlGn
VMAX   = 1.0   # ±100% relative change maps to the extremes
norm   = mcolors.TwoSlopeNorm(vmin=-VMAX, vcenter=0, vmax=VMAX)
plot_df['color'] = plot_df['rel_change'].clip(-VMAX, VMAX).map(
    lambda v: CMAP(norm(v)))

fig, ax = plt.subplots(figsize=(22, 12), dpi=100)
world.plot(ax=ax, color='#e8e8e8', edgecolor='white', linewidth=0.4)
ax.set_facecolor('#d0e8f0')
ax.set_xlim(-180, 180)
ax.set_ylim(-75, 80)

# Draw edges (anti-meridian aware)
for _, row in plot_df.iterrows():
    x1, y1, x2, y2 = row['lon0'], row['lat0'], row['lon1'], row['lat1']
    color, lw = row['color'], row['lw']
    if abs(x2 - x1) > 180:
        # Split at anti-meridian
        sign = 1 if x2 < x1 else -1
        frac = (180*sign - x1) / ((x2 + 360*sign) - x1 - (x2 - x1))
        y_mid = y1 + frac * (y2 - y1)
        ax.plot([x1, 180*sign], [y1, y_mid], color=color, lw=lw, alpha=0.85, solid_capstyle='round')
        ax.plot([-180*sign, x2], [y_mid, y2], color=color, lw=lw, alpha=0.85, solid_capstyle='round')
    else:
        ax.plot([x1, x2], [y1, y2], color=color, lw=lw, alpha=0.85, solid_capstyle='round')

# Colourbar
sm = cm.ScalarMappable(cmap=CMAP, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical', fraction=0.018, pad=0.01,
                    ticks=[-1, -0.5, 0, 0.5, 1])
cbar.ax.set_yticklabels(['−100%', '−50%', '0%', '+50%', '+100%'], fontsize=11)
cbar.set_label('Relative change in cargo weight\n(scenario B vs baseline)', fontsize=12)

# Width legend
for frac, label in [(0.2,'Low'), (0.6,'Med'), (1.0,'High')]:
    ax.plot([], [], color='grey', lw=MIN_W + (MAX_W-MIN_W)*frac**0.5, label=label)
ax.legend(title='Traffic volume', loc='lower left', fontsize=10,
          title_fontsize=10, framealpha=0.9)

ax.set_title(f'Shipping Network: {LABEL_B} vs {LABEL_A}\n'
             f'Edge width = avg traffic · Edge colour = relative change',
             fontsize=15, fontweight='bold')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2. Chokepoint Dumbbell Chart ──────────────────────────────────────────────
# Each chokepoint: two dots (A and B) connected by a coloured arrow.
# Sorted by scenario-A throughput descending.

choke = pd.merge(
    choke_a[['choke_name','cargo_total_weight']].rename(columns={'cargo_total_weight':'weight_a'}),
    choke_b[['choke_name','cargo_total_weight']].rename(columns={'cargo_total_weight':'weight_b'}),
    on='choke_name', how='outer'
).fillna(0).sort_values('weight_a', ascending=True)

choke['delta']    = choke['weight_b'] - choke['weight_a']
choke['pct']      = choke['delta'] / (choke['weight_a'] + 1e6) * 100
choke['color']    = choke['delta'].map(lambda d: '#2ecc71' if d > 0 else '#e74c3c')

fig, ax = plt.subplots(figsize=(12, max(7, len(choke) * 0.42)))
y_pos = np.arange(len(choke))

# Connecting line
for i, row in enumerate(choke.itertuples()):
    ax.plot([row.weight_a/1e9, row.weight_b/1e9], [i, i],
            color=row.color, lw=2.0, alpha=0.7, zorder=1)

# Dots
ax.scatter(choke['weight_a']/1e9, y_pos, color='steelblue', s=80,
           zorder=5, label=LABEL_A, edgecolors='white', linewidths=0.8)
ax.scatter(choke['weight_b']/1e9, y_pos, color='darkorange', s=80,
           zorder=5, label=LABEL_B, edgecolors='white', linewidths=0.8,
           marker='D')

# Delta annotations
for i, row in enumerate(choke.itertuples()):
    x_ann = max(row.weight_a, row.weight_b) / 1e9
    sign  = '+' if row.delta >= 0 else ''
    ax.text(x_ann + 0.02, i, f'{sign}{row.pct:.1f}%',
            va='center', fontsize=8, color=row.color)

ax.set_yticks(y_pos)
ax.set_yticklabels(choke['choke_name'], fontsize=9)
ax.set_xlabel('Annual cargo throughput (billion metric tons)', fontsize=12)
ax.set_title(f'Chokepoint Throughput: {LABEL_A} vs {LABEL_B}',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.set_xlim(left=0)
plt.tight_layout()
plt.show()

# Print summary table
print(f'\n{"Chokepoint":<28} {LABEL_A:>12}  {LABEL_B:>12}  {"Change":>10}  {"Δ%":>8}')
print('-'*76)
for row in choke.sort_values('weight_a', ascending=False).itertuples():
    sign = '+' if row.delta >= 0 else ''
    print(f'{row.choke_name:<28} {row.weight_a/1e9:>11.2f}B  {row.weight_b/1e9:>11.2f}B  '
          f'{row.delta/1e9:>+10.2f}B  {sign}{row.pct:>6.1f}%')

In [ ]:
# ── 3. Top Edge Gainers & Losers ──────────────────────────────────────────────
# Only edges with meaningful traffic; labelled by nearest port name if available.

sig = active[active['ships_a'] >= MIN_SHIPS].copy()

# Build a lookup: node_id -> label (port name or choke name or node id)
node_label = {}
for n, d in G.nodes(data=True):
    if d.get('source') == 'port':
        node_label[n] = d.get('portname', str(n))
    elif d.get('source') == 'choke_point':
        node_label[n] = d.get('name', str(n))
    else:
        node_label[n] = str(n)

def edge_label(e0, e1):
    l0 = node_label.get(e0, str(e0))[:18]
    l1 = node_label.get(e1, str(e1))[:18]
    return f'{l0} ↔ {l1}'

sig['label'] = sig.apply(lambda r: edge_label(r['e0'], r['e1']), axis=1)
sig['pct_change'] = sig['rel_change'] * 100

gainers = sig.nlargest(TOP_EDGES_BAR, 'abs_change')
losers  = sig.nsmallest(TOP_EDGES_BAR, 'abs_change')

fig, (ax_g, ax_l) = plt.subplots(1, 2, figsize=(20, 8))

for ax, df, title, color in [
    (ax_g, gainers, f'Top {TOP_EDGES_BAR} Gaining Edges', '#27ae60'),
    (ax_l, losers,  f'Top {TOP_EDGES_BAR} Losing Edges',  '#c0392b'),
]:
    df = df.sort_values('abs_change', ascending=(color=='#c0392b'))
    y  = np.arange(len(df))
    bars = ax.barh(y, df['abs_change']/1e6, color=color, alpha=0.8,
                   edgecolor='black', linewidth=0.3)
    ax.set_yticks(y)
    ax.set_yticklabels(df['label'], fontsize=8)
    ax.set_xlabel('Absolute change in cargo weight (million metric tons)', fontsize=11)
    ax.set_title(f'{title}\n({LABEL_B} vs {LABEL_A})', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    # Annotate % change
    for bar, (_, row) in zip(bars, df.iterrows()):
        w = bar.get_width()
        sign = '+' if row['pct_change'] >= 0 else ''
        ax.text(w + abs(w)*0.02, bar.get_y() + bar.get_height()/2,
                f'{sign}{row["pct_change"]:.0f}%', va='center', fontsize=7.5)

plt.suptitle('Biggest Route-Level Changes', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Port Impact Map ────────────────────────────────────────────────────────
# Bubble map: size = abs change in cargo, colour = % change (RdYlGn)

ports = pd.merge(
    port_a[['port_name','cargo_total_weight']].rename(columns={'cargo_total_weight':'weight_a'}),
    port_b[['port_name','cargo_total_weight']].rename(columns={'cargo_total_weight':'weight_b'}),
    on='port_name', how='outer'
).fillna(0)
ports['delta']    = ports['weight_b'] - ports['weight_a']
ports['pct']      = ports['delta'] / (ports['weight_a'] + 1e6) * 100

# Attach node lat/lon via port name
pname_to_latlon = {}
for n, d in G.nodes(data=True):
    if d.get('source') == 'port':
        pname_to_latlon[d.get('portname', '')] = (d['lon'], d['lat'])

ports['lon'] = ports['port_name'].map(lambda n: pname_to_latlon.get(n, (np.nan,np.nan))[0])
ports['lat'] = ports['port_name'].map(lambda n: pname_to_latlon.get(n, (np.nan,np.nan))[1])
ports = ports.dropna(subset=['lon','lat'])
ports = ports[ports['delta'].abs() > 1e6]  # only meaningful changes

fig, ax = plt.subplots(figsize=(22, 11), dpi=100)
world.plot(ax=ax, color='#e8e8e8', edgecolor='white', linewidth=0.4)
ax.set_facecolor('#d0e8f0')
ax.set_xlim(-180, 180)
ax.set_ylim(-75, 80)

MAX_ABS_PCT = max(ports['pct'].abs().quantile(0.95), 10)
norm_p   = mcolors.TwoSlopeNorm(vmin=-MAX_ABS_PCT, vcenter=0, vmax=MAX_ABS_PCT)
max_abs  = ports['delta'].abs().max()

sc = ax.scatter(
    ports['lon'], ports['lat'],
    s   = 20 + 300 * (ports['delta'].abs() / max_abs),
    c   = ports['pct'].clip(-MAX_ABS_PCT, MAX_ABS_PCT),
    cmap= cm.RdYlGn, norm=norm_p,
    edgecolors='black', linewidths=0.4, alpha=0.85, zorder=5
)

cbar = fig.colorbar(sc, ax=ax, orientation='vertical', fraction=0.018, pad=0.01)
cbar.set_label('% change in cargo throughput', fontsize=12)

# Size legend
for frac, lbl in [(0.2,'Small'), (0.6,'Medium'), (1.0,'Large')]:
    ax.scatter([], [], s=20+300*frac, color='grey', alpha=0.7,
               edgecolors='black', linewidths=0.4, label=lbl)
ax.legend(title='Abs. change', loc='lower left', fontsize=10,
          title_fontsize=10, framealpha=0.9)

ax.set_title(f'Port-Level Cargo Impact: {LABEL_B} vs {LABEL_A}\n'
             f'Bubble size = magnitude of change · Colour = % change',
             fontsize=15, fontweight='bold')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. KPI Summary Bar Chart ──────────────────────────────────────────────────
# High-level comparison of total cargo, ships completed, route diversity.

# Aggregate totals from edge statistics
def kpis(edge_df, choke_df):
    active_edges = (edge_df['ship_count'] > 0).sum()
    total_weight = edge_df['cargo_total_weight'].sum() / 2  # each edge counted twice
    total_value  = edge_df['cargo_total_value'].sum()  / 2
    total_ships  = edge_df['ship_count'].max()  # approx; use choke or port for true count
    choke_weight = choke_df['cargo_total_weight'].sum()
    return dict(active_edges=active_edges, total_weight_B=total_weight,
                total_value_T=total_value/1e12, choke_weight_B=choke_weight/1e9)

k_a = kpis(edge_a, choke_a)
k_b = kpis(edge_b, choke_b)

metrics = [
    ('Active edges',              k_a['active_edges'],    k_b['active_edges'],    ''),
    ('Total cargo (B tons)',      k_a['total_weight_B']/1e9, k_b['total_weight_B']/1e9, 'B t'),
    ('Choke throughput (B tons)', k_a['choke_weight_B'],  k_b['choke_weight_B'],  'B t'),
    ('Total value ($T)',          k_a['total_value_T'],   k_b['total_value_T'],   '$T'),
]

fig, axes = plt.subplots(1, len(metrics), figsize=(16, 5))
for ax, (label, val_a, val_b, unit) in zip(axes, metrics):
    bars = ax.bar([LABEL_A, LABEL_B], [val_a, val_b],
                  color=['steelblue', 'darkorange'],
                  edgecolor='black', linewidth=0.5, width=0.5)
    for bar, val in zip(bars, [val_a, val_b]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.02,
                f'{val:,.1f}{unit}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    delta_pct = (val_b - val_a) / (val_a + 1e-9) * 100
    sign = '+' if delta_pct >= 0 else ''
    ax.set_title(f'{label}\n({sign}{delta_pct:.1f}%)', fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(val_a, val_b) * 1.25)
    ax.tick_params(labelsize=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.suptitle(f'Key Performance Indicators: {LABEL_A} vs {LABEL_B}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()